In [7]:
import pandas as pd
import pyreadstat
from pathlib import Path
from typing import List

In [8]:
cols = [
    'wt_final',
    'wt_time',
    'Reg9',
    'xStrata',
    'Relig7',
    'LondInOut',
    'LA_2023',
    'Age9',
    'nchild',
    'nadult',
    'Disab3',
    'Gend3',
    'Educ6',
    'Orient4',
    'disty1_POP',
    'disty10_POP',
    'disty11_POP',
    'disty12_POP',
    'disty13_POP',
    'disty2_POP',
    'disty3_POP',
    'disty4_POP',
    'disty5_POP',
    'disty6_POP',
    'disty7_POP',
    'disty8_POP',
    'disty9_POP',
    'Eth2',
    'Eth7',
    'health',
    'NSSEC4',
    'NSSEC5',
    'NSSEC8',
    'IMD10',
    'IMD10_GR2',
    'IMD10_GR3',
    'IMD10_GR5',
    'IMD4',
    'WorkStat10',
    'WorkStat5',
    'WorkStat7',
    'Equalities_Metric_2024_GR2',
    'BMIG',
    'DVBMI',
    'DVHeightM',
    'DVWeightKG',
    'fruitveg',
    'MEMS7_ALL',
    'MEMS7GR_ALL',
    'MEMS7_SPORTCOUNT_A01',
    'MEMS7_OUT_SPORTCOUNT_A01',
    'MEMS7_IN_SPORTCOUNT_A01',
    'DURATION_SPORTCOUNT_A01',
    'VolAny',
    'VolCnt',
    'VolFrqB_Pop',
    'volint1',
    'volint2',
    'volint3',
    'volint4',
    'volint5',
    'volint6',
    'volint7',
    'Motiva_POP',
    'motivb_POP',
    'motivc_POP',
    'motivd_POP',
    'motive_POP',
    'motivex2a',
    'motivex2b',
    'motivex2c',
    'motivex2d',
    'inclus_a',
    'inclus_b',
    'inclus_c',
    'anxious',
    'comm1',
    'comm2',
    'happy',
    'indev',
    'indevtry',
    'lifesat',
    'lone',
    'worthw',
    'workactlvl',
    'WHOWITHA_SPORTCOUNT_A01',
    'WHOWITHB_SPORTCOUNT_A01',
    'WHOWITHC_SPORTCOUNT_A01',
    'WHOWITHD_SPORTCOUNT_A01',
    'limfreti1',
    'limfreti2',
    'limfreti3',
    'limfreti4',
    'limfreti5',
    'limfreti6',
    'limfreti7',
    'limfreti8'
]

In [9]:
def query_col(col_name : str, meta : pyreadstat.metadata_container) -> str:
    names = meta.column_names
    idx = names.index(col_name)
    return meta.column_labels[idx]

def get_df(path: Path, cols):
    if 'LondInOut' not in cols:
        cols.append('LondInOut')
    df , meta = pyreadstat.read_sav(path, usecols = cols)
    df = df[df['LondInOut'].notna()]
    return df, meta

def retrieve_clean(cols : List[str], data_path :str, table_dict : dict) -> pd.DataFrame:
    master_df = []
    verify_check = verify_paths(data_path, table_dict)
    if len(verify_check) > 0:
        raise FileNotFoundError(f'FilesNotFound:\n{verify_check}')
    print('All Files Found!')
    for k, v in table_dict.items():

        data_pathf = data_path.format(
            a = k,
            b = v['start'],
            c = v['end'],
            d = v['range'],
            e = v['year_num'],
            f = v['fid']
        )
        full_path = Path(data_pathf)
        df, meta = get_df(full_path, cols)
        df['year'] = f'{v['start']} - {v['end']}'
        master_df.append(df)
        print(f'Retrieved data for: {v['start']} - {v['end']}')
    combined = pd.concat(master_df)
    return combined, meta

def verify_paths(path : str, table_dict : dict) -> list:
    filesnotfound = []
    for k, v in table_dict.items():
        pathf = path.format(
            a = k,
            b = v['start'],
            c = v['end'],
            d = v['range'],
            e = v['year_num'],
            f = v['fid']
        )
  
        outcome = Path(pathf).exists()
        if outcome == False:
            filesnotfound.append(pathf)
        continue
    return filesnotfound

table_dict = {
    9288 : {
        'start' : 2022,
        'end' : 2023,
        'range' : '22-23',
        'year_num' : 'year_8',
        'fid' : '20250103'

    },
    9136 : {
        'start' : 2021,
        'end' : 2022,
        'range' : '21-22',
        'year_num' : 'year_7',
        'fid' : '20250103'
    },
    8993 : {
        'start' : 2020,
        'end': 2021,
        'range' : '20-21',
        'year_num' : 'year_6',
        'fid' : '20250103'
    },
    8899 : {
        'start' : 2019,
        'end': 2020,
        'range' : '19-20',
        'year_num' : 'year_5',
        'fid' : '20250103'
    },
    8652 : {
        'start' : 2018,
        'end': 2019,
        'range': '18-19',
        'year_num' : 'year_4',
        'fid' : '20250103'
    },
    8651 : {
        'start' : 2017,
        'end' : 2018,
        'range' : '17-18',
        'year_num' : 'year_3',
        'fid' : '20250103'
    },
    8391 : {
        'start' : 2016,
        'end' : 2017,
        'range' : '16-17',
        'year_num' : 'year_2',
        'fid' : '20250106'
    },
    8223 : {
        'start' : 2015,
        'end' : 2016, 
        'range' : '15-16',
        'year_num' : 'year_1',
        'fid' : '20250106'
    }
}
master_path = 'C:/Masters/London Sport/{a}_ActiveLifeSurvey_{b}_{c}/UKDA-{a}-spss/spss/spss28/active_lives_survey_nov_{d}_data_{e}_shared_{f}.sav'  

In [10]:
df = retrieve_clean(cols, master_path, table_dict)

All Files Found!
Retrieved data for: 2022 - 2023
Retrieved data for: 2021 - 2022
Retrieved data for: 2020 - 2021
Retrieved data for: 2019 - 2020
Retrieved data for: 2018 - 2019
Retrieved data for: 2017 - 2018
Retrieved data for: 2016 - 2017
Retrieved data for: 2015 - 2016


In [13]:
df, meta = df

In [16]:
len(df.columns)

98

In [ ]:
# df.to_csv('master_csv.csv', index=False)